# tomato — retrieval baselines

Every row through the **same scorer with the same flags**, so a difference between
two rows is the retriever and nothing else.

| family | arm | cost |
|---|---|---|
| lexical | BM25 | CPU, minutes |
| dense | BGE-large, Qwen3-Embedding, SPECTER2, SciNCL | GPU, minutes each |
| reasoning-trained dense | ReasonIR-8B | GPU, bf16, ~16 GB of weights |
| graph | G-Reasoner, GFM-RAG | **trained here**, section 5c — GPU, real training time |
| **ours** | multi-view scorer, and the same scorer + graph | section 5e — read from the ablation notebook's output, nothing trained here |

The last family is **not** a baseline. It is there because the gap between the two `ours`
rows is the graph's whole contribution on top of the learned scorer, which no pair of
baseline rows can show.

**Slices are `all`, `same`, `cross` only.** `similar`/`dissimilar` are defined by a
reference dense run, and in a baseline table that is circular: the arm defining the
slice would be graded on its own definition.

**Pooling is explicit.** SPECTER2 and SciNCL are CLS-pooled. `SentenceTransformer`
mean-pools any checkpoint that ships no ST config, which silently evaluates a
different model and understates both, so this notebook passes `--pooling cls` for
them and lets the others use their own config.

**Two disclosures that belong in the table caption, not in a footnote:**

* the SPECTER2 row is `allenai/specter2_base`, the base encoder **without** the retrieval
  adapters (`proximity` for candidate papers, ad-hoc-query for short queries). It is
  labelled `SPECTER2-base` for that reason and must not be shortened to "SPECTER2";
* SPECTER2 and SciNCL are both trained on `title [SEP] abstract`, and this corpus is
  already flattened to `"Title. Abstract"` with the boundary unrecoverable. Both see the
  same text through a slightly different input format from their papers'. It applies
  equally to both rows, so it cannot flip their comparison, but neither row is that
  paper's published number.

## 1. GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os, sys, json, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE   = "/content/drive/MyDrive/cargo-gfmrag"
DATASET = "tomato"
SPLIT   = "test"
BUNDLE  = f"{DRIVE}/{DATASET}_bundle.zip"
OUT_ROOT = f"{DRIVE}/outputs/baselines/{DATASET}"
SCIGRAPHIR_ROOT = "/content/scigraphir"
os.environ["SCIGRAPHIR_ROOT"] = SCIGRAPHIR_ROOT
os.environ["SCIGRAPHIR_DATASET"] = DATASET
S4 = f"{SCIGRAPHIR_ROOT}/experiments"
os.makedirs(OUT_ROOT, exist_ok=True)
print("bundle  ", BUNDLE)
print("writes  ", OUT_ROOT)

## 2. Unpack the bundle + install
Sections 1-4 read only the corpus and the eval scripts. **Section 5 also reads the graphs**, so run this against the FULL bundle if you want the graph rows; a `--slim` bundle carries no graphs and section 5 will say so.

In [ ]:
import zipfile, shutil
assert os.path.exists(BUNDLE), f"missing {BUNDLE}"
if os.path.isdir(SCIGRAPHIR_ROOT): shutil.rmtree(SCIGRAPHIR_ROOT)
os.makedirs(SCIGRAPHIR_ROOT, exist_ok=True)
zipfile.ZipFile(BUNDLE).extractall(SCIGRAPHIR_ROOT)

# A code_overlay on Drive wins over the bundle's copies, so a script edited after
# the bundle was built does not need a 400 MB re-upload to take effect.
OV = f"{DRIVE}/code_overlay"
if os.path.isdir(OV):
    shutil.copytree(OV, SCIGRAPHIR_ROOT, dirs_exist_ok=True); print("applied code_overlay")

CORPUS = f"{SCIGRAPHIR_ROOT}/retriever/data/{DATASET}_{SPLIT}/raw"
for f in ("documents.json", f"{SPLIT}.json"):
    assert os.path.exists(f"{CORPUS}/{f}"), f"missing {CORPUS}/{f}"
BL = f"{S4}/eval/baselines_sir4.py"
assert os.path.exists(BL), f"missing {BL} -- rebuild the bundle or use code_overlay"

_c = json.load(open(f"{CORPUS}/documents.json")); _q = json.load(open(f"{CORPUS}/{SPLIT}.json"))
_g = [len(x.get("supporting_documents") or []) for x in _q]
import collections
print(f"corpus {len(_c):,} docs | {len(_q):,} queries | golds/query {sum(_g)/len(_g):.2f}")
print("strata:", dict(collections.Counter(x.get("stratum") for x in _q)))

# Sections 1-4 are advertised as needing no graph engine, so they cannot assume the
# engine cell in 5a has run. Colab ships transformers but NOT sentence-transformers,
# and every dense arm imports it -- without this the "no GPU engine needed" claim is
# only true if you happen to run the sections out of order.
# PINNED, not latest. Colab resolves `transformers` to the newest release, which is now
# 5.x; the engine in section 5 and ReasonIR's remote-code architecture are both 4.x-era.
# An unpinned install has already produced a transformers 5.x / wandb 0.28 environment
# here. Sections 1-4 only need transformers, but the pin has to be the same one section 5a
# uses or whichever cell ran last decides the environment.
!pip -q install rank_bm25 sentence-transformers "transformers>=4.52.4,<5"
import transformers
print("ready | transformers", transformers.__version__)
assert transformers.__version__.startswith("4."), (
    f"transformers {transformers.__version__} is outside the supported 4.x range; "
    "restart the runtime after the pinned install so the 4.x wheel is the one imported")

## 3. The arms

Each instruction-aware baseline uses its documented stock query instruction rather
than a SciGraphIR-specific prompt. This keeps the rows recognizable as off-the-shelf
baselines. BM25, SPECTER2 and SciNCL take no instruction.

In [ ]:
# Qwen3's STOCK retrieval instruction from its official model card. The newline is
# part of the format and must reach the tokenizer as a real newline, not the two
# characters backslash+n. The run cell below therefore passes argv as a list.
QWEN_INSTRUCT = ("Instruct: Given a web search query, retrieve relevant passages "
                  "that answer the query\nQuery:")
BGE_INSTRUCT = "Represent this sentence for searching relevant passages: "

# ReasonIR's documented default is an empty instruction. Keep the variable explicit so
# a future task-specific ReasonIR experiment cannot silently change the stock baseline.
REASONIR_INSTRUCT = ""

# (label, tag, model, pooling, instruct, extra flags)
ARMS = [
    ("BM25",            "bm25",     "bm25",                       "auto", "",           ""),
    ("BGE-large",       "bge",      "BAAI/bge-large-en-v1.5",     "st",   BGE_INSTRUCT, ""),
    ("Qwen3-Embedding", "qwen3",    "Qwen/Qwen3-Embedding-0.6B",  "st",   QWEN_INSTRUCT, ""),
    # CLS, NOT MEAN. Both are CLS-pooled; ST's mean-pooling fallback would evaluate a
    # different model than either paper and understate them.
    # SPECTER2-base, NOT SPECTER2. The published retrieval model is this base encoder PLUS
    # a task adapter (proximity for candidate papers, ad-hoc-query for short queries).
    # Loading the base alone is a different, weaker model, so the row is labelled for what
    # it is. To make it the real thing: pip install adapters, load allenai/specter2 onto
    # the base with load_as="proximity", which baselines_sir4.py does not yet support.
    ("SPECTER2-base",   "specter2", "allenai/specter2_base",      "cls",  "",           ""),
    # NO [SEP]. SciNCL and SPECTER2 are both trained on "title [SEP] abstract"; this corpus
    # is already flattened to "Title. Abstract" in documents.json and the title boundary is
    # not recoverable from it without the pre-flattening source. Both rows therefore see a
    # slightly different input format from the one their papers used. Same text, same
    # tokeniser, one missing separator token, and it applies equally to both rows, so it
    # cannot flip their comparison -- but it is a reason not to read either as that paper's
    # published number, and it belongs in the table caption.
    ("SciNCL",          "scincl",   "malteos/scincl",             "cls",  "",           ""),
    # 8B: half precision or it does not fit, and a custom architecture so it needs remote
    # code. BFLOAT16, not float16: the model card's own usage is torch_dtype="auto", which
    # resolves to the bf16 the checkpoint was trained in. Forcing fp16 re-quantises to a
    # format with a much smaller exponent range, which is the standard way an 8B scores
    # below its published numbers. bf16 needs Ampere or newer (A100 yes, T4 no).
    ("ReasonIR-8B",     "reasonir", "reasonir/ReasonIR-8B",       "st",   REASONIR_INSTRUCT,
     "--trust-remote-code --dtype bfloat16 --batch 8"),
]
for lab, tag, m, p, ins, extra in ARMS:
    print(f"  {lab:18} {m}")

## 4. Run every arm
Skips an arm whose predictions already exist **and were produced by this exact configuration**, so a disconnect costs only the arm that was running.

The filename carries only the tag, dataset and split, so it cannot tell a mean-pooled run from a CLS-pooled one, or an old instruction from the current one. Each prediction file therefore gets a sidecar manifest holding the model, pooling, instruction, extra flags, top-k and a hash of the scorer script, and the skip is conditional on that manifest matching. A file with no manifest, or a manifest that disagrees, is re-run rather than presented as current.

In [ ]:
import hashlib, shlex
TOPK = 300   # in the signature below, so a change to it invalidates the cache

# Set True only to accept prediction files that carry no manifest (everything produced
# before this cell existed). They are then reported as though they matched the current
# configuration, which is the exact failure the manifest exists to prevent.
ACCEPT_UNVERIFIED = False

def sh(cmd, cwd=None):
    # Shell strings remain supported for the scorer cells below. Baseline inference uses
    # an argv list so instructions, especially Qwen3's required newline, arrive verbatim.
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end="")
    return p.wait()

# The scorer script itself is part of the configuration: a change to pooling, instruction
# handling or normalisation inside it changes the numbers without changing any flag.
_SCORER = hashlib.md5(open(f"{S4}/eval/baselines_sir4.py", "rb").read()).hexdigest()[:8]

def arm_sig(model, pool, ins, extra):
    return hashlib.md5(json.dumps(
        {"model": model, "pooling": pool, "instruct": ins, "extra": extra,
         "topk": TOPK, "scorer": _SCORER, "dataset": DATASET, "split": SPLIT},
        sort_keys=True).encode()).hexdigest()[:12]

PRED = {}
for lab, tag, model, pool, ins, extra in ARMS:
    dest = f"{S4}/data/predictions_{tag}_{DATASET}_{SPLIT}.json"
    man  = dest + ".manifest.json"
    sig  = arm_sig(model, pool, ins, extra)
    PRED[lab] = dest
    # RESTORE BEFORE SKIPPING. {S4} is the unpacked bundle under /content and does not
    # survive a runtime reset, so "is the file here" is the wrong question: it recomputes
    # an arm whose result is already sitting on Drive. Re-running the unpack cell has the
    # same effect. Copying back first makes a disconnect cost nothing but the running arm.
    for a, b in ((f"{OUT_ROOT}/{os.path.basename(dest)}", dest),
                 (f"{OUT_ROOT}/{os.path.basename(man)}",  man)):
        if not os.path.exists(b) and os.path.exists(a):
            os.makedirs(os.path.dirname(b), exist_ok=True); shutil.copy(a, b)
    if os.path.exists(dest):
        print(f"[restore] {lab}: recovered from Drive")
    _have = None
    if os.path.exists(man):
        try: _have = json.load(open(man)).get("sig")
        except Exception: _have = None
    if os.path.exists(dest):
        if _have == sig:
            print(f"[skip] {lab}: manifest matches this configuration"); continue
        if _have is None and ACCEPT_UNVERIFIED:
            print(f"[skip] {lab}: NO MANIFEST, accepted because ACCEPT_UNVERIFIED"); continue
        why = "no manifest" if _have is None else f"manifest {_have} != {sig}"
        print(f"[stale] {lab}: {why} -- re-running, the old file will be overwritten")
    cmd = [sys.executable, "-u", "eval/baselines_sir4.py",
           "--dataset", DATASET, "--split", SPLIT, "--model", model,
           "--pooling", pool, "--tag", tag, "--topk", str(TOPK)]
    if ins:
        cmd += ["--instruct", ins]
    if extra:
        cmd += shlex.split(extra)
    cmd += ["--out", dest]
    rc = sh(cmd, S4)
    # NOT an assert: one unavailable checkpoint (a gated repo, an OOM on the 8B)
    # should cost that row, not the whole table.
    if rc != 0:
        print(f"!! {lab} FAILED rc={rc} -- its row will be reported as missing")
    else:
        # Manifest written only on success, and only after the predictions exist, so a
        # half-written file can never be certified as current.
        json.dump({"sig": sig, "model": model, "pooling": pool, "instruct": ins,
                   "extra": extra, "topk": TOPK, "scorer_md5": _SCORER},
                  open(man, "w"), indent=1)
        shutil.copy(dest, f"{OUT_ROOT}/{os.path.basename(dest)}")
        shutil.copy(man,  f"{OUT_ROOT}/{os.path.basename(man)}")

## 5. Graph baselines — G-Reasoner and GFM-RAG

**Everything below this point is optional.** Sections 1-4 are the lexical and dense
table and need no GPU engine; stop here if that is all you want. These two arms
train a GNN, so they install the engine and take real GPU time.

Both are the published architectures trained from **random init** on our graph and
split. No pretrained GFM-RAG or G-Reasoner checkpoints are downloaded.

| arm | model | config | dataset class | supervision | ranker |
|---|---|---|---|---|---|
| G-Reasoner | `gfm_reasoner.GraphReasoner` | `sft_training` (stock) | `GraphIndexDataset` | document | n/a |
| GFM-RAG | `gfm_rag_v1.GNNRetriever` | `sft_training_gfmrag` | `GraphIndexDatasetV1` | entity | `idf_topk_ranker` |

**Both run on `tomato_train` / `tomato_test`, the OpenIE graph**, so the two rows
differ in the model and not in the substrate. They cannot both run on v16sc: GFM-RAG's
forward path needs `target_type: entity`, and v16sc has no entity nodes.

Each arm keeps its own reference recipe, which is what a baseline means. That does
leave three differences that belong to the architectures rather than to the setup, and
they should be stated as description, not hidden:

* documents are scored directly as graph nodes by G-Reasoner, and via the ranker from
  entity scores by GFM-RAG;
* `use_ent_emb: early-late-fusion` exists only for G-Reasoner;
* seed-node weighting (`init_nodes_weight`) exists only for GFM-RAG.

Both are evaluated on the same **document** metrics, so the rows compare directly even
though the losses sit at different levels of the graph.

Neither uses the handcrafted scorer, the learned scorer, the gate or any fusion. That is the
point of having them: whatever a fusion arm gains over these rows is what the
project added. Note the fusion arms run on **v16sc**, so that gain includes the graph
construction as well as the model. The construction on its own is isolated by the
separate v1-fusion versus v16-fusion pair, not by this table.

**This needs the FULL bundle**, not `--slim`, because a slim bundle carries no graphs.

### 5a. Engine
Reused verbatim from the fusion notebook, so the two builders cannot end up installing different engines.

In [ ]:
import os, sys, torch
!rm -rf /content/gfm-rag
!cd /content && unzip -q {DRIVE}/gfm-rag-adapted.zip
!pip install -q --no-deps -e /content/gfm-rag
# TORCH IS DELIBERATELY NOT IN THIS LIST. Colab ships a torch/torchvision pair built
# against each other, and asking pip for `torch` can move torch off the version its
# torchvision was compiled for. THAT mismatch is what produced
#   ImportError: cannot import name 'VideoReader'
# from datasets' torch formatter. gfmrag installs --no-deps, so nothing here needs a
# torch newer than the host image's.
!pip install -q torch-geometric sentence-transformers transformers hydra-core omegaconf \
              easydict ninja faiss-cpu pymetis wandb tqdm numpy pandas python-dotenv \
              langchain-community 2>&1 | tail -3

# REPAIR, NEVER REMOVE. This used to be `pip uninstall -y torchvision`, on the reasoning
# that gfmrag does not use it. That reasoning has expired: current transformers resolves
# PreTrainedModel through a lazy module that imports torchvision, so deleting it turns
# every `from transformers import ...` into
#   ModuleNotFoundError: Could not import module 'PreTrainedModel'
# and takes sentence-transformers, and therefore every encoder in this notebook, down with
# it. Verify the pair instead, and only intervene if it is actually broken.
import torch
try:
    import torchvision
    from transformers import PreTrainedModel          # the import that has to work
    print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | transformers ok")
except Exception as _e:
    # STOP, DO NOT SELF-HEAL. The obvious repair, `pip install torchvision`, resolves to
    # the LATEST torchvision and drags torch up with it: measured on 2026-08-17 it took a
    # stock runtime from torch 2.11.0+cu128 to 2.13.0+cu130, a 2 GB download that rebuilt
    # the whole CUDA stack, broke Colab's cudf/cuml/raft pins, and left the running kernel
    # holding the OLD torch. Silently re-pinning CUDA under a training run is far worse
    # than refusing, because the damage only surfaces as `torch.cuda.is_available()` going
    # False, or as numerics nobody can reproduce.
    #
    # A matched pair is what the stock image already ships. Getting back to it is one
    # menu action, and no pip incantation is more reliable than that.
    raise RuntimeError(
        f"torchvision/transformers are broken in this runtime "
        f"({type(_e).__name__}: {_e}).\n"
        f"torch here is {torch.__version__}.\n"
        f"FIX: Runtime > Disconnect and DELETE runtime (not 'Restart session' -- a restart "
        f"keeps whatever pip did), then run this notebook from the top. The stock image "
        f"ships a matched torch/torchvision pair and this cell installs neither, so the "
        f"check above will pass with no downloads.\n"
        f"Do NOT `pip install torchvision` to get past this: it upgrades torch and CUDA "
        f"underneath you."
    ) from _e

# THE OTHER HALF OF THE TORCHVISION STORY. datasets' torch formatter runs
# `from torchvision.io import VideoReader` whenever torchvision is importable, and
# current torchvision has REMOVED VideoReader. That import sits on the training
# dataloader's hot path, so with torchvision present every training run dies on its
# first batch. Uninstalling torchvision was the old workaround and now breaks
# transformers instead (see above), so the remaining move is to patch datasets ON
# DISK: guard the import, skip the isinstance when it is unavailable. The training
# subprocess re-imports datasets from disk, so the patch reaches it with no restart.
import re as _re
import datasets.formatting.torch_formatter as _dtf
_p = _dtf.__file__
_s = open(_p).read()
if "VideoReader = None" in _s:
    print("datasets torch formatter already patched")
else:
    _s2 = _re.sub(r'^( *)from torchvision\.io import VideoReader$',
                  lambda m: (f"{m.group(1)}try:\n{m.group(1)}    from torchvision.io import VideoReader\n"
                             f"{m.group(1)}except Exception:\n{m.group(1)}    VideoReader = None"),
                  _s, flags=_re.M)
    _s2 = _s2.replace("isinstance(value, VideoReader)",
                      "(VideoReader is not None and isinstance(value, VideoReader))")
    assert _s2 != _s, "VideoReader import not found -- datasets layout changed, patch by hand"
    open(_p, "w").write(_s2)
    print("patched datasets torch formatter:", _p)
# Replay the exact failing path in a fresh interpreter: torchvision imported (that is
# what arms the buggy branch), then a torch-formatted Dataset read.
import subprocess as _sp
_r = _sp.run([sys.executable, "-c",
              "import torchvision, datasets\n"
              "d = datasets.Dataset.from_dict({'x': [1, 2, 3]}).with_format('torch')\n"
              "print('datasets formatter ok:', d[:2]['x'])"],
             capture_output=True, text=True)
print(_r.stdout.strip())
assert _r.returncode == 0, _r.stderr[-2000:]

def soft_import(path, line, fallback):
    t = open(path).read()
    if f"try:\n    {line}" not in t:
        open(path, "w").write(t.replace(line, f"try:\n    {line}\nexcept Exception:\n    {fallback}"))
soft_import("/content/gfm-rag/gfmrag/text_emb_models/__init__.py",
            "from .qwen3_model import Qwen3TextEmbModel", "Qwen3TextEmbModel = None")
# pylate/ColBERT entity-linker is unused by SFT training; make its import non-fatal so the
# training subprocess (fresh Python re-imports gfmrag) does not crash on `import pylate`.
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/entity_linking_model/__init__.py",
            "from .colbert_el_model import ColbertELModel", "ColbertELModel = None")
# 4c. LLM-OpenIE model imports langchain_community (ChatOllama/ChatLlamaCpp); the installed version
#     dropped ChatOllama. SFT training never builds an index, so make this import non-fatal.
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/openie_model/__init__.py",
            "from .llm_openie_model import LLMOPENIEModel", "LLMOPENIEModel = None")
# 4d. same for the LLM-NER model (separate __init__, separate import line).
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/ner_model/__init__.py",
            "from .llm_ner_model import LLMNERModel", "LLMNERModel = None")
# the adapted zip strips config/wandb/ (and sometimes config/text_emb_model/) -> create before writing
for _cd in ["/content/gfm-rag/gfmrag/workflow/config/wandb",
            "/content/gfm-rag/gfmrag/workflow/config/text_emb_model"]:
    os.makedirs(_cd, exist_ok=True)
open("/content/gfm-rag/gfmrag/workflow/config/wandb/default.yaml", "w").write(
    'enabled: false\nlog_model: false\nproject: "gfm-rag"\nentity: null\nname: null\ngroup: null\ntags: []\nnotes: ""\n')
open("/content/gfm-rag/gfmrag/workflow/config/text_emb_model/qwen3_st.yaml", "w").write(
    '_target_: gfmrag.text_emb_models.BaseTextEmbModel\n'
    'text_emb_model_name: /content/qwen3\nnormalize: True\nbatch_size: 32\n'   # LOCAL path, not hub name (see cell 3)
    'query_instruct: "Instruct: Given a scientific research problem or open need, retrieve papers whose method, mechanism, or technique could be borrowed as inspiration, including transfers from other domains.\\nQuery: "\n'
    'passage_instruct: null\nmodel_kwargs: null\n')
sys.path.insert(0, "/content/gfm-rag")
# stub the unused pylate/ColBERT dep so the IN-KERNEL gfmrag import does not crash
import types as _t
for _m in ["pylate", "pylate.indexes", "pylate.models", "pylate.retrieve"]:
    sys.modules.setdefault(_m, _t.ModuleType(_m))
class _D:
    def __init__(self, *a, **k): pass
sys.modules["pylate.indexes"].PLAID = _D; sys.modules["pylate.models"].ColBERT = _D; sys.modules["pylate.retrieve"].ColBERT = _D
from gfmrag.models.gfm_reasoner import GraphReasoner
assert torch.cuda.is_available(), "Use an A100/high-RAM GPU"
print("G-Reasoner OK |", torch.cuda.get_device_name(0))

# ---- PATCH: per-epoch STRATIFIED metrics ----
# Training runs in a subprocess, so we edit the source: wrap trainer.evaluate() to add
# per-slice document_hits@k/mrr keys. _log_metrics then prints them EACH EPOCH in the same
# format as the aggregate lines. Toggle with STRAT_EVAL=0; BGE split from STRAT_BGE.
STF = "/content/gfm-rag/gfmrag/workflow/sft_training.py"
_src = open(STF).read()
if "_evaluate_stratified" not in _src:
    _inject = "\n".join([
        "    # --- injected: per-epoch stratified eval (ZERO extra forward pass) ---",
        "    # A forward hook records each eval query's gold-document rank during evaluate()'s",
        "    # existing pass; we then add per-slice keys to the metrics dict (logged as usual).",
        "    import os as _o, json as _j",
        "    from collections import defaultdict as _dd",
        "    _rec = []",
        "    _recording = {'on': False}",
        "    def _hook(_module, _inp, _out):",
        "        if not _recording['on'] or len(_inp) < 2:",
        "            return",
        "        try:",
        "            _g, _b = _inp[0], _inp[1]",
        "            _did = _g.nodes_by_type['document']",
        "            _dp = _out[:, _did]",
        "            _tgt = _b['target_nodes_mask'][:, _did].bool()",
        "            _rk = _dp.argsort(dim=-1, descending=True).argsort(dim=-1)",
        "            _ids = _b['id']",
        "            for _qi in range(_dp.shape[0]):",
        "                _pos = _tgt[_qi].nonzero(as_tuple=True)[0]",
        "                if len(_pos):",
        "                    _r = int(_rk[_qi, _pos].min().item()) + 1",
        "                    _q = _ids[_qi]",
        "                    _q = _q.item() if hasattr(_q, 'item') else _q",
        "                    _rec.append((_q, _r))",
        "        except Exception:",
        "            pass",
        "    trainer.model.register_forward_hook(_hook)",
        "    _orig_evaluate = trainer.evaluate",
        "    def _evaluate_stratified():",
        "        _rec.clear(); _recording['on'] = True",
        "        m = _orig_evaluate()",
        "        _recording['on'] = False",
        "        if _o.environ.get('STRAT_EVAL','1') != '1':",
        "            return m",
        "        try:",
        "            _name = _o.environ.get('STRAT_NAME','eval')",
        "            _tj = _o.environ.get('STRAT_TEST','')",
        "            _bp = _o.environ.get('STRAT_BGE','')",
        "            _meta = {q['id']: q for q in _j.load(open(_tj))} if _tj and _o.path.exists(_tj) else {}",
        "            _BGE = {r['id']: r for r in _j.load(open(_bp))} if _bp and _o.path.exists(_bp) else {}",
        "            def _brank(qid, g):",
        "                for i,(d,_s) in enumerate(_BGE.get(qid,{}).get('predictions',{}).get('document',[]),1):",
        "                    if d==g: return i",
        "                return 10**9",
        "            _sl = _dd(lambda: _dd(list)); _seen = set()",
        "            for _q,_r in _rec:",
        "                if _q in _seen: continue",
        "                _seen.add(_q)",
        "                _mq = _meta.get(_q, {})",
        "                _gg = _mq.get('supporting_documents') or []",
        "                _gd = _gg[0] if isinstance(_gg,list) and _gg else _gg",
        "                _st = 'same' if _mq.get('stratum')=='same' else 'cross'",
        "                _sim = 'dissim' if _brank(_q,_gd)>100 else 'sim'",
        "                for _nm in [_st,_sim] + (['cross+dissim'] if (_st=='cross' and _sim=='dissim') else []):",
        "                    _d = _sl[_nm]",
        "                    for _k in (1,5,10): _d['hits@'+str(_k)].append(float(_r<=_k))",
        "                    _d['mrr'].append(1.0/_r if _r<=100 else 0.0)",
        "            for _nm,_d in _sl.items():",
        "                _n = len(_d['mrr']) or 1",
        "                for _c in ('hits@1','hits@5','hits@10','mrr'):",
        "                    m[_name+'/document_'+_c+'/'+_nm] = sum(_d[_c])/_n",
        "        except Exception as _e:",
        "            print('[stratified] skipped:', _e)",
        "        return m",
        "    trainer.evaluate = _evaluate_stratified",
        "    trainer.train()",
    ])
    _src = _src.replace("    trainer.train()", _inject, 1)
    open(STF, "w").write(_src)
    print("patched sft_training.py -> per-epoch stratified eval")
else:
    print("stratified patch already applied")
# ---- PATCH: tqdm shows per-component RUNNING-AVERAGE losses (bce / pcr / mse / total) ----
# Patches base_trainer.py ON DISK so the training SUBPROCESS shows every loss, not just the total.
_bt = "/content/gfm-rag/gfmrag/trainers/base_trainer.py"
_bs = open(_bt).read()
_OLD = 'progress_bar.set_postfix(loss=step_metrics.get("loss", 0.0))'
_NEW = ('_names = {"bce_loss": "bce", "pcr_loss": "pcr", "mse_loss": "mse", "loss": "tot"}\n'
        '                progress_bar.set_postfix({_names.get(k, k): f"{np.mean(v):.3f}" for k, v in epoch_metrics.items()})')
if "_names.get(k, k)" in _bs:
    print("base_trainer.py already patched (per-component postfix present)")
elif _OLD in _bs:
    open(_bt, "w").write(_bs.replace(_OLD, _NEW))
    print("patched base_trainer.py -> tqdm shows running-average bce/pcr/mse/tot")
else:
    print("WARN: postfix line not found in base_trainer.py (file may have changed); inspect ~line 424")


### 5a-i. Re-pin the two packages the engine installs unbounded

In [ ]:
# The engine is 4.x-era transformers and 0.18.x-era wandb. An unpinned install here has
# already produced transformers 5.x with wandb 0.28 in this notebook.
!pip -q install "transformers>=4.52.4,<5" "wandb>=0.18.5,<0.19"
import transformers, wandb
print("transformers", transformers.__version__, "| wandb", wandb.__version__)
assert transformers.__version__.startswith("4."), transformers.__version__
assert wandb.__version__.startswith("0.18."), wandb.__version__
# Both asserts, not a print. A silent mismatch here surfaces much later as an unrelated
# traceback inside the training subprocess, which is the worst place to debug it.
print("if either assert fired, restart the runtime and re-run this cell before 5b")

In [ ]:
import os, shutil
QDIR = "/content/qwen3"
DRIVE_QWEN = f"{DRIVE}/qwen3-embedding-0.6b"
BASE = "https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main"
TOK = os.environ.get("HF_TOKEN", "")
AUTH = f'-H "Authorization: Bearer {TOK}"' if TOK else ""
NEED = ["model.safetensors","config.json","config_sentence_transformers.json","modules.json",
        "tokenizer.json","tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]

def ready(d):
    return (all(os.path.exists(f"{d}/{f}") for f in NEED)
            and os.path.getsize(f"{d}/model.safetensors") > 1_000_000_000
            and os.path.getsize(f"{d}/tokenizer.json") > 11_000_000)

if not ready(QDIR) and ready(DRIVE_QWEN):
    print("restoring Qwen3 from Drive cache ..."); shutil.copytree(DRIVE_QWEN, QDIR, dirs_exist_ok=True)

if not ready(QDIR):
    os.makedirs(f"{QDIR}/1_Pooling", exist_ok=True)
    # 1) big weight via aria2c (run ONCE — a repeat can delete the finished file)
    if not (os.path.exists(f"{QDIR}/model.safetensors") and os.path.getsize(f"{QDIR}/model.safetensors") > 1_000_000_000):
        os.system("apt-get -qq install -y aria2")
        os.system(f'aria2c -x16 -s16 -k1M --max-tries=5 --retry-wait=2 --file-allocation=none '
                  f'{AUTH} -d {QDIR} -o model.safetensors "{BASE}/model.safetensors"')
    # 2) small files bypass Xet — plain curl
    for f in ["config.json","config_sentence_transformers.json","modules.json",
              "tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]:
        os.system(f'curl -sSL -f {AUTH} "{BASE}/{f}" -o "{QDIR}/{f}"')
    # 3) tokenizer.json (11 MB, also Xet) — retry until a request lands on the good CDN
    for i in range(20):
        os.system(f"rm -f {QDIR}/tokenizer.json")
        os.system(f'curl -sSL -f {AUTH} "{BASE}/tokenizer.json" -o {QDIR}/tokenizer.json')
        if os.path.exists(f"{QDIR}/tokenizer.json") and os.path.getsize(f"{QDIR}/tokenizer.json") > 11_000_000:
            print(f"tokenizer.json ok on try {i+1}"); break
    assert ready(QDIR), "Qwen3 incomplete — re-run this cell (aria2c may need another pass)"
    os.makedirs(os.path.dirname(DRIVE_QWEN), exist_ok=True)
    shutil.copytree(QDIR, DRIVE_QWEN, dirs_exist_ok=True); print("cached Qwen3 to Drive")

# load by LOCAL PATH — never by hub name again
from sentence_transformers import SentenceTransformer
_m = SentenceTransformer(QDIR)
print("Qwen3 loaded offline:", _m.encode(["test"], normalize_embeddings=True).shape)  # (1, 1024)
del _m

### 5a-ii. Fix the vendored PyG version check

In [ ]:
# The ULTRA layers vendored in the engine parse the PyG version as:
#     pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]
# Colab resolves torch-geometric to versions like 2.6.1.post1, and int("post1") raises
# ValueError inside the first message-passing call. Taking the first three dotted
# components and keeping only the numeric ones handles "2.6.1.post1" and
# "2.7.0+pt24cu121" alike, and unlike a version pin it will not drift at the next release.
import os, torch_geometric
OLD = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]'
NEW = ('pyg_version = [int(i) for i in torch_geometric.__version__.split(".")[:3] '
       'if i.isdigit()]')
hits = []
for root, _, files in os.walk("/content/gfm-rag/gfmrag"):
    for f in files:
        if not f.endswith(".py"):
            continue
        fp = os.path.join(root, f)
        t = open(fp).read()
        if OLD in t:
            open(fp, "w").write(t.replace(OLD, NEW))
            hits.append(fp)
parsed = [int(i) for i in torch_geometric.__version__.split(".")[:3] if i.isdigit()]
if hits:
    for h in hits:
        print("  patched", h)
else:
    # IDEMPOTENT. A bare `assert hits` failed on every re-run of this cell, because the
    # first run already replaced the only occurrences. Absent is fine as long as the
    # CORRECTED form is what is there instead; absent with neither form present means the
    # engine changed and the patch would be silently doing nothing.
    _already = [os.path.join(r, f)
                for r, _, fs in os.walk("/content/gfm-rag/gfmrag") for f in fs
                if f.endswith(".py") and NEW in open(os.path.join(r, f)).read()]
    assert _already, ("neither the original nor the corrected version check is present -- "
                      "the engine changed, do not run on an unpatched copy")
    for h in _already:
        print("  already patched", h)
print(f"torch_geometric {torch_geometric.__version__} now parses to {parsed}")

### 5a-iii. Restore `torchvision.io.VideoReader` for `datasets`

Recent torchvision removed the legacy video API. `datasets`' torch formatter still does
`from torchvision.io import VideoReader` whenever torchvision is in `sys.modules`, and
`transformers` puts it there, so **every batch fetch** in the training subprocess raises

    ImportError: cannot import name 'VideoReader' from 'torchvision.io'

after the Qwen3 index is built, i.e. after the expensive part. This is a different fault
from the torch/torchvision mismatch the engine cell guards: there torchvision is broken,
here it imports fine and one symbol is gone, so deleting the runtime does not help.

The name is only needed for an `isinstance` check against tensor data, so a placeholder
class is enough. It downloads nothing, which is the point: `pip install torchvision`
resolves to the latest and drags torch and CUDA up with it.

In [ ]:
# THE TRAINING SUBPROCESS IS A FRESH PYTHON. `python -m gfmrag.workflow.sft_training` does
# not inherit this kernel's patched modules, so the shim has to live in a file that the
# subprocess imports. sft_training.py is that file, and is already patched twice below.
import torch, torchvision, torchvision.io, datasets
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | "
      f"datasets {datasets.__version__} | VideoReader "
      f"{hasattr(torchvision.io, 'VideoReader')}")

_SHIM = """# PATCH (notebook): datasets' torch formatter imports torchvision.io.VideoReader
# whenever torchvision is in sys.modules. Recent torchvision removed the legacy video API, so
# that import raises inside every batch fetch. The name is only needed for an isinstance
# check against tensor data, so a placeholder is sufficient and downloads nothing.
import torchvision.io as _tvio
if not hasattr(_tvio, "VideoReader"):
    class _NoVideoReader:
        def __init__(self, *a, **k):
            raise RuntimeError("torchvision video API removed; this shim exists only so "
                               "the datasets torch formatter can import the name")
    _tvio.VideoReader = _NoVideoReader
"""

_STF = "/content/gfm-rag/gfmrag/workflow/sft_training.py"
_t = open(_STF).read()
if "_NoVideoReader" in _t:
    print("[skip] sft_training.py already carries the shim")
else:
    # PREPENDED, not injected at an anchor. The other two patches of this file search for a
    # line inside main(); this one must run before `datasets` is imported anywhere, so it goes
    # above every import. Prepending also leaves their anchors untouched.
    open(_STF, "w").write(_SHIM + _t)
    print("shimmed", _STF)

# This kernel too: the in-kernel gfmrag import and any in-notebook dataset use hit the same
# formatter.
exec(_SHIM)
print("VideoReader present now:", hasattr(torchvision.io, "VideoReader"))

# SECOND ENGINE FIX, same cell so the ALL-mode harvest carries both. The zip's
# base_trainer calls _load_checkpoint from _setup_model BEFORE the AMP scaler is built,
# so any resume_from_checkpoint dies on self.scaler.load_state_dict with a bare
# AttributeError. The optimizer load directly above it already guards with hasattr;
# this line was missed. Skipping is harmless: bf16 runs a DISABLED GradScaler, so the
# saved scaler state is empty anyway.
_bt = "/content/gfm-rag/gfmrag/trainers/base_trainer.py"
_t = open(_bt).read()
_OLDS = 'self.scaler.load_state_dict(state["scaler"])'
if 'hasattr(self, "scaler")' in _t:
    print("[skip] base_trainer scaler guard already present")
else:
    assert _OLDS in _t, "scaler load line not found -- engine changed, inspect by hand"
    _t = _t.replace(_OLDS,
        'self.scaler.load_state_dict(state["scaler"]) if hasattr(self, "scaler") else '
        'logger.warning("resume: scaler not built yet, skipped scaler state")', 1)
    open(_bt, "w").write(_t)
    print("patched base_trainer.py scaler guard (resume path)")

# THIRD ENGINE FIX, the other half of the resume path. _setup_model loads the checkpoint
# BEFORE the bf16 cast, so Optimizer.load_state_dict casts Adam's exp_avg to the params'
# then-current fp32; after the cast the first step mixes fp32 state with bf16 grads and
# dies in _foreach_lerp_. Move the load to after precision setup, matching the dtype
# layout a fresh run creates lazily.
_t = open(_bt).read()
_EARLY = ("        if self.args.resume_from_checkpoint:\n"
          "            self._load_checkpoint(self.args.resume_from_checkpoint)\n")
_SCALER = ("        self.scaler = torch.amp.GradScaler(\n"
           "            self.device.type, enabled=self.enable_grad_scaler\n"
           "        )\n")
if _t.index(_EARLY) > _t.index(_SCALER):
    print("[skip] resume already loads after precision setup")
else:
    assert _t.count(_EARLY) == 1 and _t.count(_SCALER) == 1, "resume/scaler anchors moved"
    _t = _t.replace(_EARLY, "", 1)
    _t = _t.replace(_SCALER, _SCALER +
        "        # PATCH (notebook): resume moved after the precision cast; loading\n"
        "        # earlier poisons Adam state dtype and the first step dies.\n" + _EARLY, 1)
    open(_bt, "w").write(_t)
    print("patched base_trainer.py resume-after-precision order")

### 5b. GFM-RAG config + dataset defaults

In [ ]:
import json, os, re, csv, collections
FILES = json.loads(r'''{"/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training_gfmrag.yaml": "# GFM-RAG v1 as a BASELINE: the published architecture, trained from random init on our\n# graph and split. No pretrained GFM-RAG weights.\n#\n# This is the repo's own reference config (config/gfm_rag/sft_training.yaml) with only the\n# adaptations our data forces. It is NOT derived from sft_training.yaml (G-Reasoner); an\n# earlier version of this file was, and claimed in a comment to differ from it in two\n# places, which was false in four.\n#\n# WHY THIS RUNS ON tomato_train / tomato_test AND NOT ON v16sc.\n#   GNNRetriever.forward ends in map_entities_to_docs, which dereferences\n#   graph.target_to_other_types (model.py:318). Only GraphIndexDatasetV1 sets that\n#   attribute, and V1 needs target_type: entity. Our SciAfford graph has no `entity` nodes at\n#   all -- its node types are limitation/method/function/mechanism/task/finding/domain +\n#   document -- so target_type: entity finds zero targets there. tomato_{train,test} is the\n#   OpenIE construction, typed entity + document, which is exactly what this model expects.\n#\n# ADAPTATIONS, all forced by the data, each one deliberate:\n#   text_emb_model  mpnet -> qwen3, and feat_dim 768 -> 1024, because that is what the\n#                   graph was indexed with. A baseline reading different features from the\n#                   arms it is compared against is not a controlled row.\n#   train/valid     our splits.\n# Everything else -- dataset class, target_type, ranker, losses, lr, entity_model width --\n# is the reference recipe.\n#\n# entity_model WIDTH STAYS AT THE REFERENCE 512. feat_dim 1024 does NOT propagate into the\n# GNN: the model projects the 1024-d features down to the GNN width itself, at\n#   rel_mlp      = nn.Linear(feat_dim, entity_model.dims[0])    model.py:59\n#   question_mlp = nn.Linear(feat_dim, entity_model.dims[0])    model.py:60\n# so feat_dim and the hidden width are independent. An earlier version of this file widened\n# the six layers to 1024 \"to match feat_dim\", which was not required and roughly quadrupled\n# the layer parameter count -- a baseline with more capacity than the published architecture\n# is not that architecture.\n#\n# CACHE SAFETY. processed_dir is {root}/{name}/processed/stage2/{fingerprint}, and the\n# fingerprint is md5(class_name + text_emb_cfgs + {use_node_feat, use_relation_feat,\n# use_edge_feat, inverse_relation_feat}). The class name differs from G-Reasoner's, so the\n# two arms' graph.pt files land in different directories and cannot overwrite each other.\nhydra:\n  run:\n    dir: outputs/qa_finetune/${now:%Y-%m-%d}/${now:%H-%M-%S}\n  searchpath:\n    - pkg://gfmrag.workflow.config\n\ndefaults:\n  - _self_\n  # GFM-RAG v1's own default. SimpleRanker would report a weaker model than the paper's\n  # under the paper's name; plain idf_ranker is the un-truncated variant, and the\n  # reference uses the top-k one.\n  - doc_ranker: idf_topk_ranker\n  - text_emb_model: qwen3\n  - wandb: default\n\nseed: 1024\ntimeout: 60\nsave_pretrained: no\nload_model_from_pretrained: null\n\ndatasets:\n  # V1, not GraphIndexDataset. Required, not preferred: see the forward-path note above.\n  _target_: gfmrag.graph_index_datasets.GraphIndexDatasetV1\n  cfgs:\n    root: ./data\n    force_reload: False\n    text_emb_model_cfgs: ${text_emb_model}\n    target_type: entity          # entity reasoning, then entity->document ranking\n    use_node_feat: False         # GFM-RAG v1 does not use node features\n    use_edge_feat: False         # nor edge features\n    use_relation_feat: True\n    inverse_relation_feat: text  # v1 prefixes \"inverse\" to relation names\n  train_names:\n    - tomato_train\n  valid_names:\n    - tomato_test\n  init_datasets: True\n  feat_dim: 1024\n  max_datasets_in_memory: 10\n  data_loading_workers: 4\n\nmodel:\n  _target_: gfmrag.models.gfm_rag_v1.GNNRetriever\n  ranker: ${doc_ranker}\n  init_nodes_weight: True\n  # MUST be set whenever init_nodes_weight is True: model.py:268 asserts on it. The\n  # previous version of this file left it null with weighting on, which raised on the\n  # first forward pass.\n  init_nodes_type: document\n  dtype: bfloat16\n  entity_model:\n    # Reference width. Do NOT raise these to feat_dim: rel_mlp/question_mlp already\n    # project 1024 -> dims[0]. See the note in the header.\n    _target_: gfmrag.models.ultra.models.QueryNBFNet\n    input_dim: 512\n    hidden_dims: [512, 512, 512, 512, 512, 512]\n    message_func: distmult\n    aggregate_func: sum\n    short_cut: yes\n    layer_norm: yes\n\n# Reference supervision: ENTITY nodes. Document scores come out of the ranker, and the\n# document metrics below are computed on them, so this row stays directly comparable with\n# G-Reasoner's document metrics even though the loss sits at a different level.\n#\n# Entity-level supervision is what makes this GFM-RAG rather than \"GNNRetriever trained\n# like G-Reasoner\". To run the matched-recipe variant instead, change both\n# target_node_type below to `document` and add G-Reasoner's MSE distillation term -- but\n# then the row must not be labelled GFM-RAG.\nlosses:\n  - name: bce_loss\n    loss:\n      _target_: gfmrag.losses.BCELoss\n      adversarial_temperature: 0.2\n    weight: 0.3\n    target_node_type: entity\n  - name: pcr_loss\n    loss:\n      _target_: gfmrag.losses.ListCELoss\n    weight: 0.7\n    target_node_type: entity\n\noptimizer:\n  _target_: torch.optim.AdamW\n  lr: 5.0e-4\n\ntrainer:\n  _target_: gfmrag.trainers.SFTTrainer\n  args:\n    _target_: gfmrag.trainers.TrainingArguments\n    train_batch_size: 8\n    num_epoch: 20\n    logging_steps: 100\n    max_steps_per_epoch: null\n    resume_from_checkpoint: null\n    do_train: true\n    do_eval: true\n    save_best_only: yes\n    # Same selection metric as the G-Reasoner arm, so neither arm is chosen on a\n    # different criterion from the other. valid_names is the test set, so this keeps the\n    # best test epoch; both arms are selected identically, so the comparison between them\n    # is unaffected.\n    metric_for_best_model: document_mrr\n    dtype: ${model.dtype}\n    split_graph_inference: false\n    split_graph_training: false\n    split_graph_partition: contiguous\n  metrics:\n    - mrr\n    - ndcg@5\n    - recall@3\n    - recall@5\n  target_types:\n    - entity\n    - document\n", "/content/gfm-rag/gfmrag/utils/qa_utils.py": "# mypy: ignore-errors\n\nimport torch\nfrom torch import distributed as dist\n\nfrom gfmrag.models.ultra import variadic\n\n\nclass DocumentRetriever:\n    \"\"\"\n    Return documents based on document ranking\n    \"\"\"\n\n    def __init__(self, docs: dict, id2doc: dict) -> None:\n        self.docs = docs\n        self.id2doc = id2doc\n\n    def __call__(self, doc_ranking: torch.Tensor, top_k: int = 1) -> list:\n        top_k_docs = doc_ranking.topk(top_k).indices\n        norm_doc_scors = mini_max_scale(doc_ranking)\n        return [\n            {\n                \"title\": self.id2doc[doc.item()],\n                \"content\": self.docs[self.id2doc[doc.item()]],\n                \"score\": doc_ranking[doc].item(),\n                \"norm_score\": norm_doc_scors[doc].item(),\n            }\n            for doc in top_k_docs\n        ]\n\n\ndef mini_max_scale(tensor):\n    return (tensor - tensor.min()) / (tensor.max() - tensor.min())\n\n\ndef entities_to_mask(entities, num_nodes):\n    mask = torch.zeros(num_nodes)\n    mask[entities] = 1\n    return mask\n\n\ndef evaluate(pred, target, metrics):\n    ranking, num_pred = pred\n    answer_ranking, num_hard = target\n    answer_ranking = answer_ranking + 1\n    metric = {}\n    for _metric in metrics:\n        if _metric == \"mrr\":\n            answer_score = 1 / ranking.float()\n            query_score = variadic.variadic_mean(answer_score, num_hard)\n        elif _metric.startswith(\"recall@\"):\n            threshold = int(_metric[7:])\n            answer_score = (answer_ranking <= threshold).float()\n            query_score = (\n                variadic.variadic_sum(answer_score, num_hard) / num_hard.float()\n            )\n        elif _metric.startswith(\"hits@\"):\n            threshold = int(_metric[5:])\n            answer_score = (ranking <= threshold).float()\n            query_score = variadic.variadic_mean(answer_score, num_hard)\n        elif _metric.startswith(\"ndcg@\"):\n            # Binary-relevance nDCG, identical to score_sir4.py's definition, so the\n            # per-epoch curve and the reported table are the same quantity. Uses\n            # answer_ranking (position in the ranked list) rather than the filtered\n            # `ranking`, because nDCG is about where a gold actually landed.\n            threshold = int(_metric[5:])\n            gain = torch.where(\n                answer_ranking <= threshold,\n                1.0 / torch.log2(answer_ranking.float() + 1.0),\n                torch.zeros_like(answer_ranking, dtype=torch.float),\n            )\n            dcg = variadic.variadic_sum(gain, num_hard)\n            # Ideal DCG: min(num_golds, k) golds sitting at positions 1..n. Built as a\n            # cumulative table and indexed, so queries with different gold counts each\n            # get their own ceiling and a 2-gold query is not penalised for having\n            # fewer golds than k.\n            disc = 1.0 / torch.log2(\n                torch.arange(1, threshold + 1, device=gain.device).float() + 1.0\n            )\n            idcg_table = torch.cat(\n                [torch.zeros(1, device=gain.device), disc.cumsum(0)]\n            )\n            idcg = idcg_table[num_hard.clamp(max=threshold)]\n            query_score = dcg / idcg.clamp(min=1e-9)\n        elif _metric == \"mape\":\n            query_score = (num_pred - num_hard).abs() / (num_hard).float()\n        else:\n            raise ValueError(f\"Unknown metric `{_metric}`\")\n\n        score = query_score.mean()\n        name = _metric\n        metric[name] = score.item()\n\n    return metric\n\n\ndef gather_results(pred, target, rank, world_size, device):\n    # for multi-gpu setups: join results together\n    # for single-gpu setups: doesn't do anything special\n    ranking, num_pred = pred\n    answer_ranking, num_target = target\n\n    all_size_r = torch.zeros(world_size, dtype=torch.long, device=device)\n    all_size_ar = torch.zeros(world_size, dtype=torch.long, device=device)\n    all_size_p = torch.zeros(world_size, dtype=torch.long, device=device)\n    all_size_r[rank] = len(ranking)\n    all_size_ar[rank] = len(answer_ranking)\n    all_size_p[rank] = len(num_pred)\n    if world_size > 1:\n        dist.all_reduce(all_size_r, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_size_ar, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_size_p, op=dist.ReduceOp.SUM)\n\n    # obtaining all ranks\n    cum_size_r = all_size_r.cumsum(0)\n    cum_size_ar = all_size_ar.cumsum(0)\n    cum_size_p = all_size_p.cumsum(0)\n\n    all_ranking = torch.zeros(all_size_r.sum(), dtype=torch.long, device=device)\n    all_num_pred = torch.zeros(all_size_p.sum(), dtype=torch.long, device=device)\n    all_answer_ranking = torch.zeros(all_size_ar.sum(), dtype=torch.long, device=device)\n    all_num_target = torch.zeros(all_size_p.sum(), dtype=torch.long, device=device)\n\n    all_ranking[cum_size_r[rank] - all_size_r[rank] : cum_size_r[rank]] = ranking\n    all_num_pred[cum_size_p[rank] - all_size_p[rank] : cum_size_p[rank]] = num_pred\n    all_answer_ranking[cum_size_ar[rank] - all_size_ar[rank] : cum_size_ar[rank]] = (\n        answer_ranking\n    )\n    all_num_target[cum_size_p[rank] - all_size_p[rank] : cum_size_p[rank]] = num_target\n\n    if world_size > 1:\n        dist.all_reduce(all_ranking, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_num_pred, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_answer_ranking, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_num_target, op=dist.ReduceOp.SUM)\n\n    return (all_ranking.cpu(), all_num_pred.cpu()), (\n        all_answer_ranking.cpu(),\n        all_num_target.cpu(),\n    )\n\n\ndef batch_evaluate(pred, target, limit_nodes=None):\n    num_target = target.sum(dim=-1)\n\n    # answer2query = functional._size_to_index(num_answer)\n    answer2query = torch.repeat_interleave(num_target)\n\n    num_entity = pred.shape[-1]\n\n    # in inductive (e) fb_ datasets, the number of nodes in the graph structure might exceed\n    # the actual number of nodes in the graph, so we'll mask unused nodes\n    if limit_nodes is not None:\n        # print(f\"Keeping only {len(limit_nodes)} nodes out of {num_entity}\")\n        keep_mask = torch.zeros(num_entity, dtype=torch.bool, device=limit_nodes.device)\n        keep_mask[limit_nodes] = 1\n        # keep_mask = F.one_hot(limit_nodes, num_entity)\n        pred[:, ~keep_mask] = float(\"-inf\")\n\n    order = pred.argsort(dim=-1, descending=True)\n\n    range = torch.arange(num_entity, device=pred.device)\n    ranking = variadic.native_scatter(\n        range.expand_as(order), order, dim=-1, reduce=\"sum\"\n    )\n\n    target_ranking = ranking[target]\n    # unfiltered rankings of all answers\n    order_among_answer = variadic.variadic_sort(target_ranking, num_target)[1]\n    order_among_answer = (\n        order_among_answer + (num_target.cumsum(0) - num_target)[answer2query]\n    )\n\n    ranking_among_answer = variadic.native_scatter(\n        variadic.variadic_arange(num_target), order_among_answer, reduce=\"sum\"\n    )\n\n    # filtered rankings of all answers\n    ranking = target_ranking - ranking_among_answer + 1\n    ends = num_target.cumsum(0)\n    starts = ends - num_target\n    hard_mask = variadic.multi_slice_mask(starts, ends, ends[-1])\n    # filtered rankings of hard answers\n    ranking = ranking[hard_mask]\n\n    return ranking, target_ranking\n"}''')
for p, c in FILES.items():
    os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, 'w').write(c)
    print('wrote', p)

# The stock configs ship with another corpus in train_names/valid_names. run_baseline
# always overrides both on the command line, but a default naming a different corpus is
# the shape of every contamination this project has hit, so rewrite rather than trust.
# THE OpenIE GRAPH, NOT v16sc. GFM-RAG's forward path dereferences
# graph.target_to_other_types, which only GraphIndexDatasetV1 sets, and V1 needs
# target_type=entity. v16sc has no `entity` nodes (its types are limitation, method,
# function, mechanism, task, finding, domain, document), so V1 finds zero targets
# there. tomato_{train,test} is the OpenIE construction, typed entity + document,
# which is what both upstream models were built for. Both graph arms run on it, so
# the two rows differ in the model and not in the substrate.
#
# NOTE this is also the corpus sections 1-4 scored against, so every row in the final
# table is over the same documents and the same queries.
TRAIN, TEST = f'{DATASET}_train', f'{DATASET}_test'
DATA_ROOT = f'{SCIGRAPHIR_ROOT}/retriever/data'
# REPLACE THE WHOLE LIST, TOLERATE THE COMMENT. The stock config writes
#   train_names: # List of training dataset names
# so a regex anchored on 'train_names:\n' never matches, and its valid_names has
# THREE entries (hotpotqa x2 + musique); replacing only the first would leave the
# trainer trying to load the other two. Both failure modes are silent without the
# asserts below.
def _set_names(t, key, name):
    m = re.search(rf'({key}:[^\n]*\n)((?:[ \t]+- [^\n]*\n)+)', t)
    assert m, f'{key} block not found in config'
    indent = re.match(r'[ \t]+', m.group(2)).group(0)
    return t[:m.start()] + m.group(1) + f'{indent}- {name}\n' + t[m.end():]

for cfg in ['/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training.yaml',
            '/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training_gfmrag.yaml']:
    t = open(cfg).read()
    t = _set_names(t, 'train_names', TRAIN)
    t = _set_names(t, 'valid_names', TEST)
    open(cfg, 'w').write(t)
    # ASSERT ON THE NAME LISTS, NOT THE WHOLE FILE. Two earlier asserts both failed
    # here in opposite directions: substring 'TRAIN in t' passes on a file the regex
    # never touched ('tomato_train' is inside 'tomato_train_v16sc'), and a whole-file
    # scan for foreign corpus names fires on this config's own COMMENTS, which
    # legitimately mention v16sc while explaining why it is not used. The requirement
    # is that each list is exactly the one intended name, so check exactly that.
    for key, want in (('train_names', TRAIN), ('valid_names', TEST)):
        m = re.search(rf'{key}:[^\n]*\n((?:[ \t]+- [^\n]*\n)+)', t)
        names = re.findall(r'- (\S+)', m.group(1))
        assert names == [want], f'{key} in {cfg} is {names}, expected [{want!r}]'
    print(f"{cfg.split('/')[-1]}: -> {TRAIN} / {TEST}")

# GraphIndexDataset reads documents.json from the GRAPH directory, not the corpus one.
import shutil
for split, g in (('train', TRAIN), ('test', TEST)):
    src = f'{SCIGRAPHIR_ROOT}/retriever/data/{DATASET}_{split}/raw/documents.json'
    dst = f'{DATA_ROOT}/{g}/raw'
    stage1 = f'{DATA_ROOT}/{g}/processed/stage1'
    # Every file the dataset classes read, checked before any GPU time is spent. The
    # split json lives in stage1 (raw_dir), documents.json in raw/ -- two different
    # directories, and missing either one fails deep inside indexing.
    for need in ('nodes.csv', 'relations.csv', 'edges.csv', f'{split}.json'):
        assert os.path.exists(f'{stage1}/{need}'), (
            f'missing {stage1}/{need} -- needs the FULL bundle, not --slim')
    types = collections.Counter(r['type'] for r in
                               csv.DictReader(open(f'{stage1}/nodes.csv')))
    # GFM-RAG targets entity nodes; without them V1 builds an empty target set and
    # trains on nothing. Cheap to check here, invisible if it is not.
    assert types.get('entity', 0) > 0 and types.get('document', 0) > 0, (
        f'{g} node types are {dict(types)} -- GFM-RAG needs entity + document, so this\n'
        f'is a SciAfford graph (v16sc-style), not the OpenIE one')
    os.makedirs(dst, exist_ok=True)
    # SAME FILE when the graph dataset IS the corpus dataset, which is the case now that
    # both arms run on tomato_{train,test}: src and dst resolve to one path and
    # shutil.copy raises SameFileError. It was a real copy only while the graph was
    # v16sc and the corpus was tomato_test, two different directories.
    if os.path.abspath(src) != os.path.abspath(f'{dst}/documents.json'):
        shutil.copy(src, f'{dst}/documents.json')
        print(f'  {g}: copied documents.json from the corpus directory')
    assert os.path.exists(f'{dst}/documents.json'), f'missing {dst}/documents.json'
    print(f'  {g}: {dict(types)} | corpus ready')

### 5c. Train both
`EPOCHS` and `BATCH` are shared by both arms. Match them to whatever the fusion arms used if these rows are to be a floor for those.

In [ ]:
import hashlib   # also imported in section 4; this cell must stand alone
EPOCHS, BATCH = 10, 2

def run_baseline(config, suffix, epochs=None, batch=None):
    """Stock upstream model: no handcrafted scorer components, no semantic scorer, no fusion."""
    epochs, batch = epochs or EPOCHS, batch or BATCH
    run_dir = f"{OUT_ROOT}/{DATASET}_{suffix}"
    # BOTH, not just the checkpoint. Training can save model_best.pth and then die in the
    # predict pass (OOM is the usual way). On the next run the arm would skip, 5d would
    # find no prediction file, and the row would be reported missing forever with nothing
    # to distinguish that from "never trained".
    _ckpt = f"{run_dir}/model_best.pth"
    # sft_training.py writes predictions_{valid_dataset_name}.json, so this name has to
    # track TEST rather than be spelled out -- it changes with the graph.
    _pred = f"{run_dir}/predictions_{TEST}.json"
    # SAME PROBLEM AS THE DENSE ARMS: the directory name carries the arm, the epochs and the
    # batch, and nothing else. It cannot distinguish a run made with the reference 512-wide
    # reasoner from one made with the 1024-wide version, or a dev-selected run from a
    # test-selected one, so a finished directory from before those changes would be adopted
    # and reported as current. The signature covers the config file itself.
    _cfgp = f"/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/{config}.yaml"
    _sig = hashlib.md5(json.dumps(
        {"cfg": hashlib.md5(open(_cfgp, "rb").read()).hexdigest(),
         "epochs": epochs, "batch": batch, "topk": TOPK,
         "train": TRAIN, "valid": [TEST]},
        sort_keys=True).encode()).hexdigest()[:12]
    _armp = f"{run_dir}/arm.json"
    _was = None
    if os.path.isfile(_armp):
        try: _was = json.load(open(_armp)).get("sig")
        except Exception: _was = None
    if os.path.isfile(_ckpt) and os.path.isfile(_pred):
        if _was == _sig:
            print(f"[skip] {run_dir} already holds a finished run of this configuration")
            return run_dir
        why = "no sig in arm.json" if _was is None else f"sig {_was} != {_sig}"
        print(f"[stale] {run_dir}: {why}\n"
              "        produced by a different configuration. NOT overwriting it and NOT "
              "reporting it:\n        move it aside and re-run this cell, or leave it and "
              "accept a missing row.")
        # None, not run_dir. Returning the path let 5d find its predictions file and put the
        # stale numbers in the table, which is the same failure as having no check at all.
        return None
    if os.path.isfile(_ckpt):
        print(f"[redo] {run_dir}: checkpoint present but no predictions -- re-running")
    os.makedirs(run_dir, exist_ok=True)
    cmd = ["python", "-u", "-m", "gfmrag.workflow.sft_training",
           "--config-path", "config/gfm_reasoner", "--config-name", config,
           "text_emb_model=qwen3_st", f"datasets.cfgs.root={DATA_ROOT}",
           "datasets.cfgs.force_reload=False",
           f"datasets.train_names=[{TRAIN}]", f"datasets.valid_names=[{TEST}]",
           # The test set is the valid set, so with save_best_only and document_mrr the kept
           # checkpoint is the best test epoch, and load_best_model_at_end (default True)
           # means the predictions come from it. Both arms are selected the same way on the
           # same split, so the comparison between them is unaffected.
           f"trainer.args.num_epoch={epochs}", f"trainer.args.train_batch_size={batch}",
           # 300, matching --topk 300 on every dense arm. At 100 the graph arms score zero
           # for a gold at rank 101-300 while the dense arms get credit for it, which leaves
           # recall@100 comparable but silently deflates graph MRR against dense MRR.
           "+trainer.args.do_predict=true", f"+trainer.args.predict_top_k={TOPK}",
           f"hydra.run.dir={run_dir}"]
    env = dict(os.environ, WANDB_MODE="disabled", HYDRA_FULL_ERROR="1",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
               # OFF. The engine cell injects a per-epoch stratified-eval hook that reads
               # STRAT_TEST for the query metadata and STRAT_BGE for the similar/dissimilar
               # split. With neither set it does not fail: `stratum` is missing for every
               # query so all of them are labelled `cross`, and no BGE ranking means all of
               # them are labelled `dissim`, so the log prints document_mrr/cross and
               # /dissim lines that are really the whole eval set under two wrong names.
               # This notebook takes its slices from the external scorer in section 6 and
               # deliberately has no similar/dissimilar, so the honest setting is off. For a
               # real per-epoch same/cross curve, set STRAT_EVAL=1 with
               # STRAT_TEST=f"{CORPUS}/{SPLIT}.json" and STRAT_NAME=TEST, and ignore the
               # sim/dissim keys unless STRAT_BGE is also pointed at a BGE prediction file.
               STRAT_EVAL="0")
    # POPPED, not merely unset. If a fusion notebook ran earlier in this same Colab
    # session these are still in os.environ, and dict(os.environ, ...) would copy them
    # straight into the baseline and quietly stop it being one.
    for k in ('HANDCRAFTED_COMPONENTS', 'HANDCRAFTED_COMPONENTS_TEST', "SEMANTIC_COMPONENTS",
              "SEMANTIC_COMPONENTS_TEST", "SEMANTIC_CKPT", "SEMANTIC_POPNET",
              "FUSION_OBJECTIVE", "FUSION_ROUTER", "FUSION_GAMMAFIX", "HARDNEG_HUB",
              "HARDNEG_RAND", "HARDNEG_GRAPH", "PER_GOLD", "AUX_W", "SEM_POP_LAMBDA",
              "CCMP", "CCMP_W", "CCMP_LR", "MISS_W_AUX", "RESID_PRIOR", "CQIG_M"):
        env.pop(k, None)
    json.dump({"arm": suffix, "config": config, "epochs": epochs, "batch": batch,
               "baseline": True, "sig": _sig, "train": TRAIN, "valid": [TEST],
               "predict_top_k": TOPK},
              open(f"{run_dir}/arm.json", "w"), indent=1)
    print(f"[baseline] {config} -> {run_dir}")
    with open(f"{run_dir}/console.log", "w") as log:
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
        rc = p.wait()
    assert rc == 0, f"{suffix} failed, exit code {rc} (-9 = killed, out of memory)"
    return run_dir

RUN_DIR_GREASONER = run_baseline("sft_training",        f"greasoner_e{EPOCHS}_b{BATCH}")
RUN_DIR_GFMRAG    = run_baseline("sft_training_gfmrag", f"gfmrag_e{EPOCHS}_b{BATCH}")

### 5d. Register the graph rows
Also accepts prediction files from runs made elsewhere: set the path and the row fills without retraining.

In [ ]:
# run_baseline returns None for a directory whose signature does not match, so a stale run
# is excluded here rather than warned about and then reported. `or '_none_'` keeps the
# f-string from producing the path "None/predictions_...".
GRAPH_ARMS = {
    "G-Reasoner": f"{globals().get('RUN_DIR_GREASONER') or '_none_'}/predictions_{TEST}.json",
    "GFM-RAG":    f"{globals().get('RUN_DIR_GFMRAG') or '_none_'}/predictions_{TEST}.json",
}
for lab, p in GRAPH_ARMS.items():
    if p and os.path.exists(p):
        PRED[lab] = p; print(f"  {lab:12} {os.path.basename(os.path.dirname(p))}")
    else:
        print(f"  {lab:12} not found -> row reported as missing, not faked ({p})")

## 5e. Ours, with and without the graph

Two rows from this project's own system, marked `ours` and **not** baselines. Without
them the table only shows what the whole system beats, not which part of it is doing the
work.

| row | what it is | graph |
|---|---|---|
| `ours: multi-view scorer` | the trained sorted-MLP scorer with the joint background matchability term, the `mlp` arm of the ablation notebook | none |
| `ours: scorer + graph` | the same scorer fused with the graph channel, `z(S_op) + γ_q·relu(z(graph))` | v16sc |

**The difference between these two rows is the graph's whole contribution**, and it is the
one comparison the graph baselines in 5c cannot give you, because they have no scorer.

Read it with one caveat stated plainly: the fused row's graph is **v16sc**, so the gap
between these two rows is the graph model *and* the v16sc construction together. Splitting
those two apart is the v1-fusion versus v16-fusion pair, not this table.

Both files come from the ablation notebook, which persists them to Drive under
`outputs/{dataset}/`. This notebook writes to `outputs/baselines/{dataset}/`, so it only
reads them and cannot overwrite anything.

In [ ]:
import glob
FUSION_ROOT = f"{DRIVE}/outputs/{DATASET}"        # the ablation notebook's namespace

# The `mlp` arm, fixed-loss objective: the proposed scorer architecture. `current` and
# `dense` are its own ablations and belong in that notebook's table, not this one.
SEM_PRED = (f"{FUSION_ROOT}/semantic/"
            f"predictions_semantic_mlp_fixedloss_{DATASET}_{SPLIT}.json")

# Set this to one run directory name if several fusion runs exist. Blank picks the only
# candidate, and refuses to guess when there is more than one -- silently averaging or
# taking the newest is how an exploratory run ends up in a paper table.
FUSION_RUN = ""

OURS = {}
if os.path.exists(SEM_PRED):
    OURS["ours: multi-view scorer"] = SEM_PRED
else:
    print(f"  no semantic predictions at {SEM_PRED}\n"
          "  -> run the semantic-scorer section of the ablation notebook, which copies them "
          "to Drive")

# NOT predictions_{TEST}.json. The fusion notebook's TEST is the SciAfford dataset
# ({DATASET}_test_v16sc), not this notebook's {DATASET}_test, so filtering on this
# notebook's name would match nothing at all. The split is instead pinned by the query-id
# check below, which is the property that actually matters and catches a dev or train dump
# whatever it happens to be called.
_cand = sorted(glob.glob(f"{FUSION_ROOT}/*/predictions_*.json"))
_cand = [p for p in _cand if "/semantic/" not in p]
if FUSION_RUN:
    _hit = [p for p in _cand if os.path.basename(os.path.dirname(p)) == FUSION_RUN]
    assert _hit, f"FUSION_RUN={FUSION_RUN!r} has no predictions file; candidates:\n" + \
                 "\n".join("  " + os.path.dirname(p) for p in _cand)
    OURS["ours: scorer + graph"] = _hit[0]
elif len(_cand) == 1:
    OURS["ours: scorer + graph"] = _cand[0]
elif _cand:
    print(f"  {len(_cand)} fusion runs found -- set FUSION_RUN to one of:")
    for p in _cand:
        print("   ", os.path.basename(os.path.dirname(p)))
    print("  the row is left out rather than guessed")
else:
    print(f"  no fusion predictions under {FUSION_ROOT}")

# QUERY IDS, not counts. These files were produced against the SciAfford dataset's copy of the
# split while sections 1-4 scored the corpus copy, and a fusion run directory can also hold a
# dump of a different split. Equal counts would pass a train-vs-test mix-up of the same size;
# the id sets cannot. A row whose ids disagree is dropped, not reported against the wrong
# denominator, because the external scorer would silently just report a smaller n.
_want = {str(q["id"]) for q in json.load(open(f"{CORPUS}/{SPLIT}.json"))}
for lab, p in sorted(OURS.items()):
    _d = json.load(open(p))
    _got = {str(k) for k in _d} if isinstance(_d, dict) else \
           {str(r.get("id")) for r in _d}
    if _got == _want:
        PRED[lab] = p
        print(f"  {lab:26} n={len(_got):,}  ids match  "
              f"{os.path.basename(os.path.dirname(p))}")
    else:
        print(f"  {lab:26} DROPPED: {len(_got):,} ids, {len(_got & _want):,} shared with the "
              f"{len(_want):,} corpus queries\n{'':28}{p}\n"
              f"{'':28}this is a different split or a different corpus")

## 6. Score every arm through the same scorer

In [ ]:
# THE PROJECT'S STANDARD ROW SET FIRST. Every fusion and ablation notebook scores
#     mrr, ndcg@5, recall@3, recall@5, recall@10, recall@25, recall@100
# (plus completeset@5 on SIR-4). This notebook previously asked for recall@1/20/50 instead,
# which are what its own paper table and grid need -- so the two tables could not be laid
# side by side: recall@3 and recall@25 were simply absent from every baseline row.
#
# `--json-out` writes ONLY the requested cols, so a missing column here is not recoverable
# later without re-scoring every arm. Ask for the union and let the scorer compute it; the
# cost is a few extra set intersections per query.
STD_COLS = ("mrr", "ndcg@5", "recall@3", "recall@5", "recall@10", "recall@25", "recall@100")
# Every cutoff the grid and the paper table below can be pointed at, for both recall and
# hits, so changing CUTOFFS or METRIC in section 7 never needs a re-score.
GRID_KS  = (1, 3, 5, 10, 20, 25, 50, 100)
# dict.fromkeys: order-preserving dedup, since recall@3 etc. appear in both lists.
COLS = ",".join(dict.fromkeys(list(STD_COLS)
                              + [f"recall@{k}" for k in GRID_KS]
                              + [f"hits@{k}" for k in GRID_KS]))
# NO completeset@5. It needs sets.json via --sets, which is not passed below, and without it
# score_sir4 returns 0.0 for the column rather than omitting it -- a row of zeros that reads
# as a measurement. TOMATO has no sets.json at all; on a SIR-4 build add both together.
print("cols:", COLS)
QS = f"{CORPUS}/{SPLIT}.json"
SCORES = {}
for lab in list(PRED):
    p = PRED[lab]
    if not p or not os.path.exists(p):
        print(f"skipping {lab}: no predictions"); continue
    jo = f"{OUT_ROOT}/scores_{lab.replace(' ', '_').replace('/', '-')}.json"
    # No --sets (CompleteSet@k is undefined here) and no --bge (all/same/cross only).
    rc = sh(f"python3 -u eval/score_sir4.py --pred {p} --queries {QS} --cols {COLS} "
            f"--name '{lab}' --json-out {jo}", S4)
    if rc == 0 and os.path.exists(jo):
        SCORES[lab] = json.load(open(jo))
print("\nscored:", list(SCORES))

## 7. The table

`CUTOFFS` picks the three (same, cross) pairs. **Change it in one line** if the paper
table uses different ones — the full grid below is printed at every cutoff so you can
read any of them off without re-running anything.

`drop` is the mean relative cross-domain drop across the three cutoffs:
`mean_k (1 - cross@k / all@k)`.

In [ ]:
# THE TWO KNOBS THAT SHAPE THE PAPER TABLE. Both are read off the full grid printed
# below, so if a column does not match the draft, change these rather than re-running:
# every cutoff and both slices are already computed.
CUTOFFS = (1, 10, 50)          # the three cutoffs, one (left, right) pair each
# same vs cross, NOT all vs cross. "all" already contains the cross queries, so an
# all-minus-cross gap compares cross-domain retrieval against a mixture that includes it
# and understates the drop. The thesis claim is about same-domain versus cross-domain.
PAIR    = ("same", "cross")    # ("all", "cross") mixes the two populations
METRIC  = "recall"             # "recall" or "hits". score_sir4 has only ndcg@5,
                               # so nDCG cannot be one of the three cutoff columns.
# LOUD, not "--". Section 6 requests recall@k and hits@k for every k in GRID_KS, so any
# cutoff drawn from that tuple is present. One outside it silently prints "--" in every row,
# which reads as "the arm did not run" rather than "you asked for a column nobody scored".
assert all(k in GRID_KS for k in CUTOFFS), (
    f"CUTOFFS {CUTOFFS} is not a subset of GRID_KS {GRID_KS}; add the cutoff there and "
    "re-run section 6, or the paper table will be empty")

def cell(sc, slc, k):
    d = sc.get(slc) or {}
    v = d.get(f"{METRIC}@{k}")
    return None if v is None else 100.0 * v

L, R = PAIR
hdr = "".join(f"  {METRIC}@{k:<3} {L[:5]:>5} {R[:5]:>5}" for k in CUTOFFS)
print(f"{'arm':20}" + hdr + "   drop")
print("-" * (20 + len(hdr) + 8))
# OURS INCLUDED. Section 5e loaded and section 6 scored those two rows; leaving them out of
# ORDER dropped them from the printed table only, which is the worst kind of omission: the
# numbers exist, the work was done, and the table silently does not show them. globals() so
# that skipping 5e still prints the baseline rows.
ORDER = ([l for l, *_ in ARMS] + list(GRAPH_ARMS)
         + list(globals().get("OURS", {})))
for lab in ORDER:
    sc = SCORES.get(lab)
    if not sc:
        print(f"{lab:20}" + "  (not run)"); continue
    row, drops = "", []
    for k in CUTOFFS:
        a, c = cell(sc, L, k), cell(sc, R, k)
        row += f"      {a:5.1f} {c:5.1f}" if a is not None and c is not None else "        --    --"
        if a and c is not None: drops.append(1.0 - c / a)
    d = f"  {-100 * sum(drops) / len(drops):5.0f}%" if drops else "     --"
    print(f"{lab:20}{row}{d}")

# THE STANDARD ROW SET, in the same order the ablation notebooks print it, so a baseline row
# and a fusion row can be read off side by side. The wider grid follows it.
print("\n\nSTANDARD METRICS (%) -- same columns as the fusion and ablation notebooks")
for slc in ("all", "same", "cross"):
    print(f"\n--- {slc} ---")
    print(f"{'arm':26}{'n':>6}" + "".join(f"{c.replace('recall@', 'R@'):>9}" for c in STD_COLS))
    for lab in ORDER:
        sc = (SCORES.get(lab) or {}).get(slc)
        if not sc:
            continue
        print(f"{lab:26}{sc.get('n', 0):>6}"
              + "".join(f"{100*sc[c]:>9.1f}" if sc.get(c) is not None else f"{'--':>9}"
                        for c in STD_COLS))

print("\n\nFULL GRID (%) -- every cutoff, for reading off a different table")
for slc in ("all", "same", "cross"):
    print(f"\n--- {slc} ---")
    print(f"{'arm':26}{'n':>6}{'MRR':>7}{'nDCG@5':>8}"
          + "".join(f"{'R@'+str(k):>8}" for k in GRID_KS))
    for lab in ORDER:
        sc = (SCORES.get(lab) or {}).get(slc)
        if not sc:
            continue
        print(f"{lab:26}{sc.get('n', 0):>6}{100*sc.get('mrr', 0):>7.1f}"
              f"{100*sc.get('ndcg@5', 0):>8.1f}"
              + "".join(f"{100*sc.get(f'recall@{k}', 0):>8.1f}" for k in GRID_KS))

json.dump(SCORES, open(f"{OUT_ROOT}/baselines_all.json", "w"), indent=1)
print(f"\nwrote {OUT_ROOT}/baselines_all.json")